In [ ]:
import os, sys, torch, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import spectrogram
from pathlib import Path
# 1. CONFIGURACIÓN DE RUTAS (Ajusta si es necesario)
REPO_ROOT   = r"c:\repos\DroneDetectionRF"
DATA_ROOT   = r"C:\TFM_data\NoisyUAV\drone_RF_data"
TEST_FILE   = r"C:\TFM_data\NoisyUAV\ground_truth_test_set.csv"
OUTPUT_DIR  = f"{REPO_ROOT}\\NoisyUAV\\modelo_v5\\outputs"
MODEL_PATH  = f"{OUTPUT_DIR}\\abmil_model.pt"
sys.path.insert(0, REPO_ROOT)
from NoisyUAV.modelo_v5.pipeline import (
    discover_files, segment_file, BurstEncoder, GatedAttentionMIL, 
    FS, IQ_INPUT_LEN, EMBED_DIM, PSD_N_BINS, BAG_MAX_INSTANCES
)

In [ ]:
# 2. SELECCIÓN DE CASO (CAMBIA ESTO PARA EXPLORAR)
# Tipos: 0, 1, 2, 3 (Drones) | 4 (Ruido/Interferencia)
TIPO_BUSCADO = 3   # Cambia a 0, 1, 2, 3 o 4
SNR_BUSCADO  = -2  # Cambia a 20, 10, 0, -5, -10, -15, -20

In [ ]:
def interactive_analysis_v2(target_class, target_snr):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sys.path.insert(0, REPO_ROOT)
    from NoisyUAV.modelo_v5.pipeline import (
        discover_files, segment_file, BurstEncoder, GatedAttentionMIL, 
        FS, BAG_MAX_INSTANCES, EMBED_DIM
    )
    # A. Buscar Fichero
    _, test_files = discover_files(DATA_ROOT, TEST_FILE)
    candidates = [f for f in test_files if f['class'] == target_class and f['snr'] == target_snr]
    if not candidates: return print("❌ No hay ficheros con esa combinación.")
    file_entry = candidates[0]
    
    # B. Modelos y Carga
    encoder = BurstEncoder().to(device)
    mil = GatedAttentionMIL().to(device)
    ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
    encoder.load_state_dict(ckpt['encoder']); mil.load_state_dict(ckpt['mil'])
    encoder.eval(); mil.eval()
    # C. Procesamiento
    storage_path = segment_file(file_entry)
    with open(storage_path, 'rb') as f: bursts = pickle.load(f)
    valid_bursts = bursts[:BAG_MAX_INSTANCES]
    
    iq_list, psd_list = [], []
    for b in valid_bursts:
        it, pt = encoder.preprocess_burst(b['iq'])
        iq_list.append(it); psd_list.append(pt)
        
    with torch.no_grad():
        emb = encoder(torch.stack(iq_list).to(device), torch.stack(psd_list).to(device))
        logits, attn = mil(emb.unsqueeze(0))
        prob, attn_w = torch.sigmoid(logits).item(), attn.squeeze(0).cpu().numpy()
    # D. VISUALIZACIÓN MEJORADA
    d = torch.load(file_entry['path'], map_location='cpu', weights_only=False)
    iq_complex = (d['x_iq'][0] + 1j * d['x_iq'][1]).numpy()
    f, t, Sxx = spectrogram(iq_complex, fs=FS, nperseg=512, noverlap=256, return_onesided=False)
    f, Sxx = np.fft.fftshift(f), np.fft.fftshift(Sxx, axes=0)
    
    fig = plt.figure(figsize=(16, 14))
    gs = fig.add_gridspec(4, 1, height_ratios=[4, 1.5, 2, 1])
    # 1. Espectrograma con Ventanas
    ax1 = fig.add_subplot(gs[0])
    ax1.pcolormesh(t*1000, f/1e6, 10*np.log10(np.abs(Sxx)+1e-12), cmap='magma', shading='gouraud')
    for i, b in enumerate(valid_bursts):
        rect = plt.Rectangle((b['t0'], f[0]/1e6), b['t1']-b['t0'], (f[-1]-f[0])/1e6, 
                             edgecolor='cyan', facecolor='cyan', alpha=attn_w[i]*0.6, lw=2)
        ax1.add_patch(rect)
        ax1.text(b['t0'], f[-1]/1e6, f"#{i}", color='cyan', fontsize=10, weight='bold')
    ax1.set_title(f"DETECCIONES V5: {Path(file_entry['path']).name} (SNR: {target_snr}dB)", fontsize=14)
    ax1.set_ylabel("Frecuencia (MHz)")
    # 2. Pesos de Atención
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.bar([ (b['t0']+b['t1'])/2 for b in valid_bursts ], attn_w, 
            width=[ b['t1']-b['t0'] for b in valid_bursts ], color='red', alpha=0.8, edgecolor='white')
    ax2.set_ylabel("Atención")
    ax2.set_title("Mecanismo de Atención (Importancia de cada ráfaga)")
    # 3. Embeddings por Ráfaga (Horizontal y Grande)
    ax3 = fig.add_subplot(gs[2])
    sns.heatmap(emb.cpu().numpy(), ax=ax3, cmap='viridis', cbar_kws={'label': 'Activación'})
    ax3.set_title(f"Mapa de Características (Embeddings) - {len(valid_bursts)} candidatos", fontsize=12)
    ax3.set_ylabel("Índice de Candidato (#)")
    ax3.set_xlabel("Dimensión de la Característica (1-128)")
    # 4. Decisión Final
    ax4 = fig.add_subplot(gs[3])
    ax4.axis('off')
    res_txt = f"RESULTADO: {'DRON' if prob > 0.5 else 'RUIDO'} | PROBABILIDAD: {prob:.2%}"
    ax4.text(0.5, 0.5, res_txt, fontsize=22, ha='center', va='center', weight='bold',
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='green' if prob > 0.5 else 'red', boxstyle='round,pad=1'))
    plt.tight_layout()
    plt.show()

In [ ]:
# Ejecutar el análisis
interactive_analysis_v2(TIPO_BUSCADO, SNR_BUSCADO)